### Data Reading

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *
dbutils.fs.ls('/Volumes/databrickstutorial/default/data/')

In [0]:
df = spark.read.csv('/Volumes/databrickstutorial/default/data/BigMart Sales.csv', header=True, inferSchema=True)

In [0]:
df.display()

In [0]:
json_df = spark.read.json('/Volumes/databrickstutorial/default/data/drivers.json')

In [0]:
json_df.display()

### DDL SCHEMA


In [0]:
df.printSchema()

### SELECT

In [0]:
df_select = df.display()

In [0]:
df.select('Item_Identifier','Item_Weight','Item_Fat_Content').display()

###ALIAS

In [0]:
df.select(df['Item_Identifier'].alias('Item_ID')).display()

###FILTER/WHERE

### Scenario - 1

In [0]:
df.filter(df['Item_Fat_Content'] =='Regular').display()

### Scenario - 2

In [0]:
df.filter( (df['Item_Type']=='Soft Drinks') & (df['Item_Weight']<10)).display()

### Scenario - 3

In [0]:
df.filter (df['Outlet_Location_Type'].isin(['Tier 1','Tier 2']) & (df['Outlet_Size'].isNull())).display()

### withColumnRenamed

In [0]:
df.withColumnRenamed('Item_Weight', 'Item_Wt').display()

### withColumn

###  Scenario - 1

In [0]:
df = df.withColumn('Item_weight_Pounds',df['Item_Weight']*2.20462)
df.display()

### Scenario - 2

In [0]:
from pyspark.sql.functions import *

df.withColumn('Item_Fat_Content',regexp_replace(df['Item_Fat_Content'],'Regular','Reg'))\
    .withColumn('Item_Fat_Content',regexp_replace(df['Item_Fat_Content'],'Low Fat','LF'))

### Type Casting

In [0]:
df = df.withColumn("Item_Weight", df["Item_Weight"].cast("double"))


In [0]:
df.printSchema()

### Sort/orderby

### Scenario - 1

In [0]:
df.sort(df['Item_Weight'].desc()).display()

### Scenario - 2

In [0]:
df.sort(df['Item_Visibility']).display()

### Scenario - 3

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending=[False,True]).display()

### Limit

In [0]:
df.limit(10).display()

### Drop

In [0]:
df = df.drop('Item_weight_Pounds')

In [0]:
df.display()

### DROP_DUPLICATES

### Scenario - 1

In [0]:
df.dropDuplicates().display()

### Scenario - 2

In [0]:
df.drop_duplicates(subset = ['Item_Type']).display()

In [0]:
df.distinct().display()

### UNION

In [0]:
data1 = [('1','Steve'),
         ('2','Balan')]
schema1 = ('id STRING, name STRING')
df1 = spark.createDataFrame(data = data1, schema = schema1)

data2 = [('3','Deva'),
         ('4','Princy')]
schema2 = ('id STRING, name STRING')
df2 = spark.createDataFrame(data = data2, schema = schema2)


In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
df1.union(df2).display()

In [0]:
data1 = [('Steve','1'),
         ('Balan','2')]
schema1 = ('name STRING, id STRING')
df1 = spark.createDataFrame(data = data1, schema = schema1)

df1.display()


In [0]:
df1.union(df2).display()

### Union by name

In [0]:
df2.unionByName(df1).display()

### String Function

In [0]:
df.select(initcap(df['Item_Type']),upper(df['Outlet_Type']),lower(df['Item_Fat_Content'])).display()

### Date Functions

#### Current_Date

In [0]:
df = df.withColumn('curr_date',current_date())
df.display()

#### Date_Add()

In [0]:
df = df.withColumn('week_afer',date_add(df['curr_date'],7))
df.display()

%md
#### Date_Sub()

In [0]:
df = df.withColumn('Week_before',date_sub(df['curr_date'],7))
df.display()

#### Date_Sub() Alternative

In [0]:
df = df.withColumn('Week_before',date_add(df['curr_date'],-7))
df.display()

#### Datediff()

In [0]:
df = df.withColumn('datediff',datediff(df['week_afer'],df['curr_date']))
df.display()

#### Date_Format()

In [0]:
df = df.withColumn('week_before',date_format(df['week_before'],'MM-dd-yyyy'))
df.display()

### Handing Nulls

#### Dropping Nulls

In [0]:
df.dropna('all').display()

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset=['Outlet_Size']).display()

#### Filling Nulls

In [0]:
df.fillna('NotAvailable').display()

In [0]:
df.fillna('NotAvaliable',subset=['Outlet_Size']).display()

## SPLIT and INDEXING

### SPLIT

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()

#### Indexing

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[0]).display()

### EXPLODE

In [0]:
df_exp = df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df_exp.display()

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

### ARRAY_CONTAINS

In [0]:
df_exp.withColumn('Type1_flag',array_contains('Outlet_Type','Type1')).display()

### Group_by

#### Scenario - 1

In [0]:
df.groupBy('Item_Type').agg(sum('Item_MRP')).display()

#### Sceniario - 2

In [0]:
df.groupBy('Item_Type').agg(avg('Item_MRP')).display()

%md
#### Sceniario - 3

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP')).display()

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP')).display()

#### Scenario - 4

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP'),avg('Item_MRP').alias('Average_MRP')).display()

### COLLECT_LIST

In [0]:
data  = [('user1', 'book1'),
         ('user1', 'book2'),
         ('user2', 'book2'),
         ('user2', 'book4'),
         ('user3', 'book1')]
schema = 'user string, book string'
df_book = spark.createDataFrame(data, schema)
df_book.display()

In [0]:
df_book.groupby('user').agg(collect_list('book')).display()

### PIVOT

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(sum('Item_MRP')).display()

### WHEN-OTHERWISE

#### Scenario - 1

In [0]:
df = df.withColumn('Food_Type',when((df['Item_Type'] == 'Meat'),'Non-Veg').otherwise('Veg'))
df.display()

#### Scenario - 2

In [0]:
df = df.withColumn('Food_Exp',when((df['Food_Type'] == 'Non-Veg'),df['Item_MRP']*1.2)\
                            .when((df['Food_Type'] == 'Veg'),df['Item_MRP']*1.1)\
                            .otherwise(df['Item_MRP']))
df.display()                            

## JOINS

In [0]:

# first DataFrame
d1 = [("1", "sravan", "d01"),
      ("2", "ojaswi", "d02"),
      ("3", "rohith", "d03"),
      ("4", "sridevi", "d03"),
      ("5", "bobby", "d05"),
      ("6", "steve", "d06")]
cols1 = ['emp_id', 'emp_name', 'dep_id']
df_table1 = spark.createDataFrame(d1, cols1)

# second DataFrame
d2 = [("d01", "45000", "IT"),
      ("d02", "145000", "Manager"),
      ("d03", "45000", "HR"),
      ("d04", "34000", "Sales"),
      ("d05", "34000", "Accounts")]
cols2 = ['dep_id', 'salary', 'department']
df_table2 = spark.createDataFrame(d2, cols2)

df_table1.display()

In [0]:
df_table2.display()

### INNER JOIN

In [0]:
df_table1.join(df_table2, df_table1.dep_id == df_table2.dep_id, "inner").display()

### LEFT JOIN

In [0]:
df_table1.join(df_table2, df_table1.dep_id == df_table2.dep_id, "left").display()

### RIGHT JOIN

In [0]:
df_table1.join(df_table2, df_table1.dep_id == df_table2.dep_id, "right").display()

### FULL JOIN

In [0]:
df_table1.join(df_table2, df_table1.dep_id == df_table2.dep_id, "full").display()

### ANTI JOIN

In [0]:
df_table1.join(df_table2, df_table1.dep_id == df_table2.dep_id, "anti").display()

In [0]:
df_table2.join(df_table1, df_table1.dep_id == df_table2.dep_id, "anti").display()

## Window Functions

### ROW NUMBER()

In [0]:
df.withColumn('rownum', row_number().over(Window.orderBy(df['Item_Identifier']))).display()

### RANK()

In [0]:
df.withColumn('rank', rank().over(Window.orderBy(df['Item_Identifier'])))\
    .withColumn('dense_rank', dense_rank().over(Window.orderBy(df['Item_Identifier']))).display()

### DENSE_RANK()

In [0]:

df.withColumn('dense_rank', dense_rank().over(Window.orderBy(desc(df['Item_Identifier'])))).display()


### Cumulative Sum

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type'))).display()

In [0]:
window_spec = (
    Window
    .partitionBy("Item_Type")
    .orderBy("Item_MRP")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
df.withColumn(
    "cumsum",
    sum("Item_MRP").over(window_spec)
).display()

In [0]:
window_spec = (
    Window
    .partitionBy("Item_Type")
    .orderBy("Item_MRP")
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)
df.withColumn(
    "cumsum",
    sum("Item_MRP").over(window_spec)
).display()


## USER DEFINED FUNCTIONS (UDF)



### STEP - 1

In [0]:
def my_func(x):
    return x*2

In [0]:
my_udf = udf(my_func)

In [0]:
df.withColumn('CalMRP',my_udf(df['Item_MRP'])).display()

## DATA WRITING

### CSV

In [0]:
#df.write.mode("overwrite").format("csv").option("header", "true").save("/Volumes/databrickstutorial/default/data/")

### APPEND

In [0]:
#df.write.format("csv").mode('append').save("/Volumes/databrickstutorial/default/data/duplicate.csv")

### OVERWRITE

In [0]:
#df.write.format("csv").mode('overwrite').option("path", "/Volumes/databrickstutorial/default/data/duplicate.csv").save()

### ERROR

In [0]:
#df.write.format("csv").mode('error').option("path", "/Volumes/databrickstutorial/default/data/duplicate.csv").save()

### IGNORE

In [0]:
#df.write.format("csv").mode('ignore').option("path", "/Volumes/databrickstutorial/default/data/duplicate.csv").save()

### PARQUET 

In [0]:
#df.write.format("parquet").mode('overwrite').option("path", "/Volumes/databrickstutorial/default/data/duplicate.csv").save()

### AsTable

In [0]:
df.write.saveAsTable("my_table")

## SPARL SQL

In [0]:
df.createTempView('my_view')

In [0]:
%sql

select * from my_view
where Item_Fat_Content ='Low Fat'